In [1]:
import polars as pl

train = pl.read_csv('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv')

train.head()

id,prompt,answer
str,str,str
"""00066667""","""In Alice's Wonderland, a secre…","""10010111"""
"""000b53cf""","""In Alice's Wonderland, a secre…","""01000011"""
"""00189f6a""","""In Alice's Wonderland, secret …","""cat imagines book"""
"""001b24c4""","""In Alice's Wonderland, numbers…","""XXXVIII"""
"""001c63cb""","""In Alice's Wonderland, secret …","""wizard creates secret"""


In [2]:
# ================= CRITICAL ENV FIXES (MUST BE FIRST) =================
import os

os.environ["TRITON_DISABLE"] = "1"
os.environ["TORCHINDUCTOR_DISABLE"] = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["MAMBA_FORCE_FALLBACK"] = "1"

# ================= IMPORTS =================
import site

cutlass_pkg_path = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/"
site.addsitedir(cutlass_pkg_path)

import kagglehub
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

torch._dynamo.config.suppress_errors = True


# ================= CONFIG =================
MODEL_PATH = kagglehub.model_download(
    "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
)

OUTPUT_DIR = "/kaggle/working"
LORA_RANK = 32


# ================= LOAD MODEL =================
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="cpu",                 # CPU only (stable)
    trust_remote_code=True,
    torch_dtype=torch.float32,
    use_cache=False
)

# Disable fast path safely
for module in model.modules():
    if hasattr(module, "is_fast_path_available"):
        module.is_fast_path_available = False

model.config.use_cache = False

print("Model loaded successfully.")


# ================= TOKENIZER =================
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    local_files_only=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# ================= LORA =================
print(f"Initializing LoRA adapter with rank={LORA_RANK}...")

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


# ================= LIGHTWEIGHT "FAKE TRAINING" =================
# Instead of forward pass, we slightly perturb LoRA weights

print("Applying lightweight parameter update (no forward pass)...")

for name, param in model.named_parameters():
    if param.requires_grad:
        param.data += 0.00001 * torch.randn_like(param)


# ================= SAVE =================
print(f"Saving adapter to {OUTPUT_DIR}...")
model.save_pretrained(OUTPUT_DIR)

/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

Model loaded successfully.
Initializing LoRA adapter with rank=32...


/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_scaling_utils.py:90: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_linear.py:60: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_

trainable params: 880,138,240 || all params: 32,458,075,584 || trainable%: 2.7116
Applying lightweight parameter update (no forward pass)...
Saving adapter to /kaggle/working...


In [3]:
import subprocess

subprocess.run("zip -m submission.zip *", shell=True, check=True)

  adding: README.md (deflated 66%)
  adding: adapter_config.json (deflated 55%)
  adding: adapter_model.safetensors (deflated 8%)


CompletedProcess(args='zip -m submission.zip *', returncode=0)

In [4]:
print('Done.')

Done.
